# GeoClip Zero-Shot Baseline

Evaluate pretrained GeoClip on the MMlandmarks query set **without fine-tuning**.
The model embeds query ground images and gallery GPS coordinates into a shared
512-dim space, then retrieves the nearest GPS by cosine similarity.

**Gallery:** configurable via `gallery.source` in [configs/geoclip_baseline.yaml](../../configs/geoclip_baseline.yaml):
- `"paper"` (default, 100,539 coords = 99,539 index-satellite + 1,000 query-landmark GPS) — matches the camera-ready MML paper Sec 5.2 protocol. Reproduces the 21.37 % @1 km row of Table 3. Because every query's GT GPS is in the gallery, this is an **upper bound**.
- `"index"` (99,539 coords) — index-satellite only. Honest in-the-wild result (~6.67 % @1 km). Per the paper author: *"21 % is a geolocalization upper limit, 6.67 % is more realistic in the wild."*

**Queries:** 18,688 query ground images (multiple images per landmark, each scored against the landmark's ground-truth GPS).

**Metric:** Accuracy @ {1, 25, 200, 750, 2500} km (Haversine distance).

## 1. Setup

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

plt.rcParams.update({"figure.dpi": 120})

# Load config
with open("../../configs/geoclip_baseline.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_ROOT = Path("../../") / cfg["data"]["root"]
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"

device = cfg["inference"]["device"] if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Data root: {DATA_ROOT.resolve()}")

Device: cuda
Data root: /dtu/blackhole/02/137570/MML


## 2. Load Model

In [2]:
from mmgeo.geolocalizations.geoclip.geoclip_baseline import (
    GeoClipBaseline,
    load_gallery_coords,
    load_query_data3,
)
from mmgeo.geolocalizations.geoclip.evaluate import (
    accuracy_at_thresholds,
    median_error,
    haversine,
)

baseline = GeoClipBaseline(device=device)

total_params = sum(p.numel() for p in baseline.model.parameters())
trainable_params = sum(p.numel() for p in baseline.model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 6467.67it/s]

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/geoclip/model/location_encoder.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sel

Total parameters: 438,050,306
Trainable parameters: 10,432,257


## 3. Build GPS Gallery

In [3]:
# Load both galleries up-front; we rebuild per-source before each inference pass.
GALLERY_SOURCES = ["paper", "index"]
galleries = {}
for source in GALLERY_SOURCES:
    coords = load_gallery_coords(DATA_ROOT, source=source)
    galleries[source] = coords
    print(
        f"source={source!r:>8}: {len(coords):>6,} GPS points · "
        f"lat [{coords[:, 0].min():.2f}, {coords[:, 0].max():.2f}] · "
        f"lon [{coords[:, 1].min():.2f}, {coords[:, 1].max():.2f}]"
    )


source= 'paper': 100,539 GPS points · lat [20.00, 49.03] · lon [-155.89, -66.93]


source= 'index': 99,539 GPS points · lat [20.00, 49.03] · lon [-155.89, -66.93]


## 4. Load Query Data

In [4]:
thing = 1
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (1000, 1000, 1000)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/16 [00:00<?, ?batch/s]

Predicting:   6%|▋         | 1/16 [00:02<00:34,  2.32s/batch]

Predicting:  12%|█▎        | 2/16 [00:04<00:28,  2.03s/batch]

Predicting:  19%|█▉        | 3/16 [00:05<00:25,  1.93s/batch]

...

Predicting:  88%|████████▊ | 14/16 [00:25<00:03,  1.83s/batch]

Predicting:  94%|█████████▍| 15/16 [00:27<00:01,  1.82s/batch]

Predicting: 100%|██████████| 16/16 [00:28<00:00,  1.65s/batch]

Predicting: 100%|██████████| 16/16 [00:28<00:00,  1.79s/batch]

[index] 1000 predictions in 28.6s
 Threshold (km) paper (%) index (%)
              1     11.90      3.30
             25     24.80     21.20
            200     44.10     42.30
            750     71.80     71.60
           2500     93.60     93.80

 paper: median error 280.7 km · mean 639.0 km
 index: median error 306.5 km · mean 643.0 km
2


In [5]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (1902, 1902, 1902)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/30 [00:00<?, ?batch/s]

Predicting:   3%|▎         | 1/30 [00:01<00:53,  1.83s/batch]

Predicting:   7%|▋         | 2/30 [00:03<00:51,  1.82s/batch]

Predicting:  10%|█         | 3/30 [00:05<00:49,  1.83s/batch]

...

Predicting:  93%|█████████▎| 28/30 [00:51<00:03,  1.81s/batch]

Predicting:  97%|█████████▋| 29/30 [00:53<00:01,  1.88s/batch]

Predicting: 100%|██████████| 30/30 [00:54<00:00,  1.74s/batch]

Predicting: 100%|██████████| 30/30 [00:54<00:00,  1.82s/batch]

[index] 1902 predictions in 54.5s
 Threshold (km) paper (%) index (%)
              1     11.88      3.47
             25     24.97     20.93
            200     43.17     41.06
            750     72.29     71.50
           2500     94.16     94.06

 paper: median error 416.8 km · mean 623.9 km
 index: median error 427.9 km · mean 639.6 km
3


In [6]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (2583, 2583, 2583)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/41 [00:00<?, ?batch/s]

Predicting:   2%|▏         | 1/41 [00:01<01:13,  1.85s/batch]

Predicting:   5%|▍         | 2/41 [00:03<01:11,  1.83s/batch]

Predicting:   7%|▋         | 3/41 [00:05<01:09,  1.84s/batch]

...

Predicting:  95%|█████████▌| 39/41 [01:11<00:03,  1.83s/batch]

Predicting:  98%|█████████▊| 40/41 [01:13<00:01,  1.83s/batch]

Predicting: 100%|██████████| 41/41 [01:14<00:00,  1.53s/batch]

Predicting: 100%|██████████| 41/41 [01:14<00:00,  1.82s/batch]

[index] 2583 predictions in 74.5s
 Threshold (km) paper (%) index (%)
              1     12.47      3.91
             25     25.05     21.18
            200     43.59     41.66
            750     72.51     71.54
           2500     93.65     93.38

 paper: median error 474.0 km · mean 634.3 km
 index: median error 479.4 km · mean 656.7 km
4


In [7]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (3104, 3104, 3104)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/49 [00:00<?, ?batch/s]

Predicting:   2%|▏         | 1/49 [00:01<01:28,  1.84s/batch]

Predicting:   4%|▍         | 2/49 [00:03<01:25,  1.82s/batch]

Predicting:   6%|▌         | 3/49 [00:05<01:29,  1.95s/batch]

...

Predicting:  96%|█████████▌| 47/49 [01:26<00:03,  1.83s/batch]

Predicting:  98%|█████████▊| 48/49 [01:28<00:01,  1.83s/batch]

Predicting: 100%|██████████| 49/49 [01:29<00:00,  1.60s/batch]

Predicting: 100%|██████████| 49/49 [01:29<00:00,  1.83s/batch]

[index] 3104 predictions in 89.5s
 Threshold (km) paper (%) index (%)
              1     12.89      4.03
             25     26.58     22.42
            200     44.30     42.14
            750     72.36     71.01
           2500     93.52     93.17

 paper: median error 483.2 km · mean 634.8 km
 index: median error 492.7 km · mean 662.1 km
5


In [8]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (3490, 3490, 3490)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/55 [00:00<?, ?batch/s]

Predicting:   2%|▏         | 1/55 [00:02<01:54,  2.12s/batch]

Predicting:   4%|▎         | 2/55 [00:03<01:43,  1.95s/batch]

Predicting:   5%|▌         | 3/55 [00:05<01:39,  1.91s/batch]

...

Predicting:  96%|█████████▋| 53/55 [01:38<00:03,  1.91s/batch]

Predicting:  98%|█████████▊| 54/55 [01:39<00:01,  1.89s/batch]

Predicting: 100%|██████████| 55/55 [01:41<00:00,  1.66s/batch]

Predicting: 100%|██████████| 55/55 [01:41<00:00,  1.84s/batch]

[index] 3490 predictions in 101.1s
 Threshold (km) paper (%) index (%)
              1     13.78      4.33
             25     28.17     23.64
            200     44.79     42.38
            750     72.69     71.20
           2500     93.09     92.78

 paper: median error 488.8 km · mean 641.4 km
 index: median error 505.1 km · mean 668.8 km
6


In [9]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (3738, 3738, 3738)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/59 [00:00<?, ?batch/s]

Predicting:   2%|▏         | 1/59 [00:01<01:47,  1.85s/batch]

Predicting:   3%|▎         | 2/59 [00:03<01:44,  1.84s/batch]

Predicting:   5%|▌         | 3/59 [00:05<01:43,  1.84s/batch]

...

Predicting:  97%|█████████▋| 57/59 [01:45<00:03,  1.88s/batch]

Predicting:  98%|█████████▊| 58/59 [01:47<00:01,  1.87s/batch]

Predicting: 100%|██████████| 59/59 [01:48<00:00,  1.58s/batch]

Predicting: 100%|██████████| 59/59 [01:48<00:00,  1.84s/batch]

[index] 3738 predictions in 108.3s
 Threshold (km) paper (%) index (%)
              1     14.37      4.60
             25     29.00     24.18
            200     45.29     42.80
            750     72.04     70.33
           2500     92.67     92.40

 paper: median error 532.4 km · mean 654.2 km
 index: median error 567.3 km · mean 683.3 km
7


In [10]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4011, 4011, 4011)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/63 [00:00<?, ?batch/s]

Predicting:   2%|▏         | 1/63 [00:01<01:55,  1.86s/batch]

Predicting:   3%|▎         | 2/63 [00:03<01:52,  1.84s/batch]

Predicting:   5%|▍         | 3/63 [00:05<01:50,  1.85s/batch]

...

Predicting:  97%|█████████▋| 61/63 [01:52<00:03,  1.82s/batch]

Predicting:  98%|█████████▊| 62/63 [01:54<00:01,  1.89s/batch]

Predicting: 100%|██████████| 63/63 [01:55<00:00,  1.71s/batch]

Predicting: 100%|██████████| 63/63 [01:55<00:00,  1.84s/batch]

[index] 4011 predictions in 116.0s
 Threshold (km) paper (%) index (%)
              1     15.13      4.71
             25     29.54     24.28
            200     45.05     42.33
            750     71.45     69.38
           2500     92.12     91.80

 paper: median error 549.4 km · mean 674.1 km
 index: median error 578.9 km · mean 709.2 km
8


In [11]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4184, 4184, 4184)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/66 [00:00<?, ?batch/s]

Predicting:   2%|▏         | 1/66 [00:01<02:01,  1.86s/batch]

Predicting:   3%|▎         | 2/66 [00:03<01:57,  1.84s/batch]

Predicting:   5%|▍         | 3/66 [00:05<01:55,  1.84s/batch]

...

Predicting:  97%|█████████▋| 64/66 [01:58<00:03,  1.85s/batch]

Predicting:  98%|█████████▊| 65/66 [02:00<00:01,  1.84s/batch]

Predicting: 100%|██████████| 66/66 [02:01<00:00,  1.54s/batch]

Predicting: 100%|██████████| 66/66 [02:01<00:00,  1.84s/batch]

[index] 4184 predictions in 121.2s
 Threshold (km) paper (%) index (%)
              1     16.09      4.85
             25     30.45     24.86
            200     44.74     41.95
            750     70.43     68.36
           2500     91.75     91.40

 paper: median error 560.2 km · mean 691.0 km
 index: median error 592.6 km · mean 727.5 km
9


In [12]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4419, 4419, 4419)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/70 [00:00<?, ?batch/s]

Predicting:   1%|▏         | 1/70 [00:01<02:07,  1.85s/batch]

Predicting:   3%|▎         | 2/70 [00:03<02:04,  1.83s/batch]

Predicting:   4%|▍         | 3/70 [00:05<02:02,  1.82s/batch]

...

Predicting:  97%|█████████▋| 68/70 [02:05<00:03,  1.84s/batch]

Predicting:  99%|█████████▊| 69/70 [02:07<00:01,  1.82s/batch]

Predicting: 100%|██████████| 70/70 [02:07<00:00,  1.37s/batch]

Predicting: 100%|██████████| 70/70 [02:07<00:00,  1.82s/batch]

[index] 4419 predictions in 127.4s
 Threshold (km) paper (%) index (%)
              1     16.41      4.89
             25     30.87     24.96
            200     45.08     42.00
            750     70.54     68.50
           2500     91.54     91.26

 paper: median error 545.8 km · mean 696.5 km
 index: median error 593.5 km · mean 731.3 km
10


In [13]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4550, 4550, 4550)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/72 [00:00<?, ?batch/s]

Predicting:   1%|▏         | 1/72 [00:01<02:11,  1.85s/batch]

Predicting:   3%|▎         | 2/72 [00:03<02:07,  1.83s/batch]

Predicting:   4%|▍         | 3/72 [00:05<02:05,  1.82s/batch]

...

Predicting:  97%|█████████▋| 70/72 [02:08<00:03,  1.82s/batch]

Predicting:  99%|█████████▊| 71/72 [02:10<00:01,  1.81s/batch]

Predicting: 100%|██████████| 72/72 [02:10<00:00,  1.38s/batch]

Predicting: 100%|██████████| 72/72 [02:10<00:00,  1.82s/batch]

[index] 4550 predictions in 130.9s
 Threshold (km) paper (%) index (%)
              1     17.30      5.45
             25     31.98     26.02
            200     45.52     42.64
            750     70.26     68.42
           2500     91.41     91.12

 paper: median error 564.7 km · mean 699.3 km
 index: median error 594.0 km · mean 732.7 km
11


In [14]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4631, 4631, 4631)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/73 [00:00<?, ?batch/s]

Predicting:   1%|▏         | 1/73 [00:01<02:14,  1.87s/batch]

Predicting:   3%|▎         | 2/73 [00:03<02:10,  1.84s/batch]

Predicting:   4%|▍         | 3/73 [00:05<02:07,  1.83s/batch]

...

Predicting:  97%|█████████▋| 71/73 [02:10<00:03,  1.82s/batch]

Predicting:  99%|█████████▊| 72/73 [02:12<00:01,  1.81s/batch]

Predicting: 100%|██████████| 73/73 [02:13<00:00,  1.51s/batch]

Predicting: 100%|██████████| 73/73 [02:13<00:00,  1.82s/batch]

[index] 4631 predictions in 133.1s
 Threshold (km) paper (%) index (%)
              1     18.12      5.83
             25     32.89     26.82
            200     45.52     42.60
            750     70.14     68.15
           2500     91.04     90.78

 paper: median error 568.6 km · mean 710.1 km
 index: median error 607.0 km · mean 745.5 km
12


In [15]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4752, 4752, 4752)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/75 [00:00<?, ?batch/s]

Predicting:   1%|▏         | 1/75 [00:01<02:16,  1.84s/batch]

Predicting:   3%|▎         | 2/75 [00:03<02:13,  1.82s/batch]

Predicting:   4%|▍         | 3/75 [00:05<02:17,  1.92s/batch]

...

Predicting:  97%|█████████▋| 73/75 [02:14<00:03,  1.82s/batch]

Predicting:  99%|█████████▊| 74/75 [02:15<00:01,  1.81s/batch]

Predicting: 100%|██████████| 75/75 [02:16<00:00,  1.45s/batch]

Predicting: 100%|██████████| 75/75 [02:16<00:00,  1.82s/batch]

[index] 4752 predictions in 136.4s
 Threshold (km) paper (%) index (%)
              1     18.52      6.08
             25     33.80     27.48
            200     46.36     43.12
            750     70.27     68.16
           2500     91.04     90.82

 paper: median error 563.3 km · mean 706.9 km
 index: median error 591.0 km · mean 742.6 km
13


In [16]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (4966, 4966, 4966)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/78 [00:00<?, ?batch/s]

Predicting:   1%|▏         | 1/78 [00:01<02:21,  1.84s/batch]

Predicting:   3%|▎         | 2/78 [00:03<02:17,  1.81s/batch]

Predicting:   4%|▍         | 3/78 [00:05<02:23,  1.91s/batch]

...

Predicting:  97%|█████████▋| 76/78 [02:20<00:03,  1.86s/batch]

Predicting:  99%|█████████▊| 77/78 [02:21<00:01,  1.85s/batch]

Predicting: 100%|██████████| 78/78 [02:22<00:00,  1.64s/batch]

Predicting: 100%|██████████| 78/78 [02:22<00:00,  1.83s/batch]

[index] 4966 predictions in 143.0s
 Threshold (km) paper (%) index (%)
              1     18.81      5.86
             25     33.91     27.51
            200     46.46     43.23
            750     70.22     68.12
           2500     91.04     90.88

 paper: median error 549.2 km · mean 705.3 km
 index: median error 591.4 km · mean 738.9 km
14


In [17]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5096, 5096, 5096)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/80 [00:00<?, ?batch/s]

Predicting:   1%|▏         | 1/80 [00:01<02:25,  1.84s/batch]

Predicting:   2%|▎         | 2/80 [00:03<02:21,  1.81s/batch]

Predicting:   4%|▍         | 3/80 [00:05<02:19,  1.81s/batch]

...

Predicting:  98%|█████████▊| 78/80 [02:23<00:03,  1.85s/batch]

Predicting:  99%|█████████▉| 79/80 [02:25<00:01,  1.84s/batch]

Predicting: 100%|██████████| 80/80 [02:26<00:00,  1.65s/batch]

Predicting: 100%|██████████| 80/80 [02:26<00:00,  1.83s/batch]

[index] 5096 predictions in 146.8s
 Threshold (km) paper (%) index (%)
              1     18.94      5.95
             25     33.95     27.39
            200     46.00     42.58
            750     70.00     67.86
           2500     90.99     90.82

 paper: median error 557.3 km · mean 710.6 km
 index: median error 596.6 km · mean 746.2 km
15


In [18]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5310, 5310, 5310)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/83 [00:00<?, ?batch/s]

Predicting:   1%|          | 1/83 [00:01<02:31,  1.84s/batch]

Predicting:   2%|▏         | 2/83 [00:03<02:27,  1.82s/batch]

Predicting:   4%|▎         | 3/83 [00:05<02:25,  1.82s/batch]

...

Predicting:  98%|█████████▊| 81/83 [02:28<00:03,  1.81s/batch]

Predicting:  99%|█████████▉| 82/83 [02:30<00:01,  1.81s/batch]

Predicting: 100%|██████████| 83/83 [02:32<00:00,  1.86s/batch]

Predicting: 100%|██████████| 83/83 [02:32<00:00,  1.84s/batch]

[index] 5310 predictions in 152.4s
 Threshold (km) paper (%) index (%)
              1     18.89      6.16
             25     33.80     27.19
            200     45.65     42.17
            750     69.60     67.38
           2500     90.85     90.70

 paper: median error 567.3 km · mean 719.9 km
 index: median error 614.3 km · mean 755.8 km
16


In [19]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5328, 5328, 5328)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/84 [00:00<?, ?batch/s]

Predicting:   1%|          | 1/84 [00:01<02:32,  1.84s/batch]

Predicting:   2%|▏         | 2/84 [00:03<02:28,  1.81s/batch]

Predicting:   4%|▎         | 3/84 [00:05<02:26,  1.81s/batch]

...

Predicting:  98%|█████████▊| 82/84 [02:30<00:03,  1.82s/batch]

Predicting:  99%|█████████▉| 83/84 [02:32<00:01,  1.81s/batch]

Predicting: 100%|██████████| 84/84 [02:32<00:00,  1.45s/batch]

Predicting: 100%|██████████| 84/84 [02:32<00:00,  1.82s/batch]

[index] 5328 predictions in 152.9s
 Threshold (km) paper (%) index (%)
              1     19.65      6.38
             25     34.76     27.97
            200     46.73     43.00
            750     69.99     67.70
           2500     90.71     90.50

 paper: median error 545.6 km · mean 715.2 km
 index: median error 611.0 km · mean 752.5 km
17


In [20]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5287, 5287, 5287)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/83 [00:00<?, ?batch/s]

Predicting:   1%|          | 1/83 [00:02<02:49,  2.06s/batch]

Predicting:   2%|▏         | 2/83 [00:03<02:34,  1.91s/batch]

Predicting:   4%|▎         | 3/83 [00:05<02:29,  1.86s/batch]

...

Predicting:  98%|█████████▊| 81/83 [02:28<00:03,  1.86s/batch]

Predicting:  99%|█████████▉| 82/83 [02:30<00:01,  1.84s/batch]

Predicting: 100%|██████████| 83/83 [02:31<00:00,  1.64s/batch]

Predicting: 100%|██████████| 83/83 [02:31<00:00,  1.83s/batch]

[index] 5287 predictions in 151.9s
 Threshold (km) paper (%) index (%)
              1     19.71      6.66
             25     35.46     28.48
            200     47.29     43.37
            750     69.91     67.71
           2500     90.94     90.64

 paper: median error 539.4 km · mean 709.4 km
 index: median error 618.8 km · mean 749.9 km
18


In [21]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5328, 5328, 5328)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/84 [00:00<?, ?batch/s]

Predicting:   1%|          | 1/84 [00:01<02:33,  1.85s/batch]

Predicting:   2%|▏         | 2/84 [00:03<02:28,  1.82s/batch]

Predicting:   4%|▎         | 3/84 [00:05<02:25,  1.80s/batch]

...

Predicting:  98%|█████████▊| 82/84 [02:30<00:03,  1.89s/batch]

Predicting:  99%|█████████▉| 83/84 [02:32<00:01,  1.86s/batch]

Predicting: 100%|██████████| 84/84 [02:33<00:00,  1.49s/batch]

Predicting: 100%|██████████| 84/84 [02:33<00:00,  1.82s/batch]

[index] 5328 predictions in 153.3s
 Threshold (km) paper (%) index (%)
              1     20.14      6.87
             25     36.15     28.92
            200     47.67     43.73
            750     69.99     67.81
           2500     90.84     90.50

 paper: median error 569.9 km · mean 711.0 km
 index: median error 603.0 km · mean 751.4 km
19


In [22]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5377, 5377, 5377)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/85 [00:00<?, ?batch/s]

Predicting:   1%|          | 1/85 [00:01<02:36,  1.86s/batch]

Predicting:   2%|▏         | 2/85 [00:03<02:31,  1.83s/batch]

Predicting:   4%|▎         | 3/85 [00:05<02:28,  1.81s/batch]

...

Predicting:  98%|█████████▊| 83/85 [02:32<00:03,  1.82s/batch]

Predicting:  99%|█████████▉| 84/85 [02:34<00:01,  1.88s/batch]

Predicting: 100%|██████████| 85/85 [02:34<00:00,  1.40s/batch]

Predicting: 100%|██████████| 85/85 [02:34<00:00,  1.82s/batch]

[index] 5377 predictions in 154.8s
 Threshold (km) paper (%) index (%)
              1     20.36      6.90
             25     36.56     29.20
            200     48.08     44.04
            750     70.19     68.07
           2500     91.00     90.72

 paper: median error 552.4 km · mean 704.8 km
 index: median error 598.0 km · mean 744.3 km
20


In [23]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")

thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]


results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
  
    p_lat, p_lon = preds[:, 0], preds[:, 1]

    distances = haversine(p_lat, p_lon, true_lat, true_lon)

    #We normalize by landmark
    distances_by_source[source] = pd.Series(distances).groupby(landmark_ids).mean().values

    results_by_source[source] = {}
    for t in thresholds:
        x = distances <= t
        # Since we want to normalize by landmark using landmark_ids, we compute the mean per landmark first, then average those means.
        x = pd.Series(x).groupby(landmark_ids).mean().values
        results_by_source[source][t] = x.mean()
    
    

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")

thing += 1
print(thing)

Query images: (5480, 5480, 5480)

[paper] build_gallery (100,539 GPS points)…


Predicting:   0%|          | 0/86 [00:00<?, ?batch/s]

Predicting:   1%|          | 1/86 [00:01<02:38,  1.86s/batch]

Predicting:   2%|▏         | 2/86 [00:03<02:33,  1.83s/batch]

Predicting:   3%|▎         | 3/86 [00:05<02:30,  1.82s/batch]

...

Predicting:  98%|█████████▊| 84/86 [02:34<00:03,  1.89s/batch]

Predicting:  99%|█████████▉| 85/86 [02:36<00:01,  1.86s/batch]

Predicting: 100%|██████████| 86/86 [02:37<00:00,  1.68s/batch]

Predicting: 100%|██████████| 86/86 [02:37<00:00,  1.83s/batch]

[index] 5480 predictions in 157.4s
 Threshold (km) paper (%) index (%)
              1     20.31      6.90
             25     36.55     29.14
            200     48.08     44.09
            750     70.35     68.25
           2500     90.93     90.75

 paper: median error 553.1 km · mean 704.5 km
 index: median error 601.9 km · mean 742.6 km
21


## 5. Run Inference

## 6. Evaluate

## 7. Visualizations

## 8. Summary for Zero Shot

Zero-shot GeoClip on MMlandmarks, 18,688 query ground images, V100. Single HPC submit
runs both gallery sources back-to-back.

### `gallery.source: paper` — paper protocol (index + query = 100,539 GPS)

Reproduces the MML paper's Table 3 off-the-shelf GeoCLIP row within rounding. Because
every query's GT GPS sits in the gallery, this is an **upper bound** on achievable
performance, not an in-the-wild number.

| Threshold (km) | Accuracy (%) |
|---------------:|-------------:|
| 1              | 21.35        |
| 25             | 36.44        |
| 200            | 48.61        |
| 750            | 71.41        |
| 2500           | 91.52        |

- **Median error:** 225.2 km
- **Mean error:** 674.6 km

### `gallery.source: index` — honest in-the-wild (99,539 GPS, no query leakage)

Same model, query GT GPS removed from the gallery. Measures what off-the-shelf GeoCLIP
can actually do on US landmark localization without gallery leakage. Per Oskar
Kristoffersen (first author): *"21 % is a geolocalization upper limit, 6.67 % is more
realistic in the wild."*

| Threshold (km) | Accuracy (%) |
|---------------:|-------------:|
| 1              |  6.67        |
| 25             | 28.79        |
| 200            | 44.48        |
| 750            | 69.07        |
| 2500           | 91.07        |

- **Median error:** 294.3 km
- **Mean error:** 724.2 km

### Paper contrast

| Method | Dataset | Gallery | @1 km | @25 km | @200 km | @750 km | @2500 km |
|---|---|---:|---:|---:|---:|---:|---:|
| GeoClip (own paper) | Im2GPS3k (global) | 100k | 14.11 | 34.47 | 50.65 | 69.67 | 83.82 |
| Off-shelf GeoClip (MML paper) | MMlandmarks (US) | 101k (index+query) | **21.37** | **36.44** | 48.57 | 71.45 | 91.50 |
| **Ours (`paper`)** | MMlandmarks (US) | 101k (index+query) | **21.35** | **36.44** | 48.61 | 71.41 | 91.52 |
| **Ours (`index`)** | MMlandmarks (US) | 100k (index only) | **6.67** | **28.79** | 44.48 | 69.07 | 91.07 |

We reproduce the MML paper row to within 0.02 points at @1 km. The gap between the two
`Ours` rows is the query-leakage effect: including query GT coordinates in the gallery
gives the model a guaranteed-correct candidate to pick, inflating all thresholds.

> An earlier run on the 17,557 train-landmark gallery scored 19.22 % @1 km. That number
> is inflated by cluster-luck — train and query landmarks co-locate in the same tourist
> cities, so the nearest train-landmark GPS is often coincidentally <1 km from a query.
> Not a fair comparison to either of the paper galleries above.

**Next steps (Phase 2):** Fine-tune the Location Encoder and linear image head on the
MMlandmarks train split. Fair improvement lives on top of the `index` baseline
(28.79 % @25 km, not 36.44 %) — the paper gallery's leakage makes it hard to beat by
model changes alone.